facts -  реальное кол-во курьеров, которое нужно было, чтобы обслужить все заказы за предыдущую неделю. Также содержит фактическую нагрузку за прошедшую неделю, число курьеров, предлагавших свои услуги (не обязательно вышедших на смены), фактическое кол-во заказов, лайфтайм даркстора, название города, общее число трудоустроенных курьеров, число курьеров, вышедших на смены, расходы на маркетинг, факт повышенного спроса.

shifts - прогноз числа заказов на неделю и запланированная нагрузка (число заказов на одного курьера в час), а также прогнозное число курьеров, чтобы обслужить эти заказы. Каждая дата - понедельник соответствующей недели, на которую приведены прогнозные значения.

train - содержит для каждого даркстора на даты в прошлом дефицит курьеров в поле target.

test.csv - тестовый датасет, упорядочен по store_id. Необходимо сделать прогноз для каждого из приведенных дарксторов НА НЕДЕЛЮ (на 24 ноября 2025г).

### FACTS
- `calendar_dt` - дата в формате YYYY-MM-DD
- `store_id` - идентификатор даркстора
- `fact_staff_value_lag_1` - фактическое число человек, необходимое, чтобы обслужить все заказы на прошлой неделе,
- `fact_load_factor_lag_1` - фактическая нагрузка на курьеров на прошлой неделе,
- `num_available_couriers_lag_1` - число курьеров, предложивших свои услуги за предыдущую неделю,
- `fact_num_orders_lag_1` - фактическое число заказов за предыдущую неделю,
- `fact_percent_lateness_lag_1` - процент опозданий за предыдущую неделю,
- `store_lifetime_in_days` - число дней функционирования даркстора,
- `city_nm` - город функционирования даркстора,
- `fact_staff_churn` - число курьеров, уволившихся из компании,
- `flag_high_load_lag_1` - признак повышенного спроса за предыдущую неделю,
- `marketing_costs_lag_1` - затраты на маркетинг, осуществленные за предыдущую неделю,
- `fact_couriers_with_shifts_lag_1` - число курьеров, выходивших на смены на предыдущей неделе.

### SHIFTS
- `calendar_dt` - дата в формате YYYY-MM-DD
- `store_id` - идентификатор даркстора
- `predicted_staff_value` - предсказанное число человек, необходимое, чтобы обслужить все заказы на неделе прогноза,
- `predicted_num_orders` - предсказанное число заказов на неделю прогноза,
- `predicted_load_factor` - предсказанная нагрузка на курьеров (число заказов в час, которое приходится на одного курьера) на неделю прогноза.

### Примечание для проверяющих
Изначальную обработку признаков я делал для Catboost, но когда закончил и решил попробовать использовать LightGBM, то оказалось, что он работает лучше. Потом Я решил попробовать использовать kfold, и получилось еще лучше. 

In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/intern-task-regression-courier-deficit-challenge/sample_submission.csv
/kaggle/input/competitions/intern-task-regression-courier-deficit-challenge/facts.csv
/kaggle/input/competitions/intern-task-regression-courier-deficit-challenge/shifts_prediction.csv
/kaggle/input/competitions/intern-task-regression-courier-deficit-challenge/train.csv
/kaggle/input/competitions/intern-task-regression-courier-deficit-challenge/test.csv


In [2]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import KFold

In [3]:
df_facts = pd.read_csv('facts.csv')
df_sample = pd.read_csv('sample_submission.csv')
df_shifts = pd.read_csv('shifts_prediction.csv')
df_test = pd.read_csv('test.csv')
df_train = pd.read_csv('train.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'facts.csv'

У файлов train, shifts и facts есть общие колонки: дата и id магазина, поэтому можно объединить датасеты, используя эти колонки как ключи. Но для начала отформатируем даты и отсортируем их 

In [ ]:
df_facts['calendar_dt'] = pd.to_datetime(df_facts['calendar_dt'])
df_shifts['calendar_dt'] = pd.to_datetime(df_shifts['calendar_dt'])
df_train['calendar_dt'] = pd.to_datetime(df_train['calendar_dt'])

df_test['calendar_dt'] = pd.to_datetime('2025-11-24')
df_test['target'] = np.nan

df = pd.concat([df_train, df_test], ignore_index=True)
df = df.merge(df_facts, on=['calendar_dt','store_id'], how='left')
df = df.merge(df_shifts, on=['calendar_dt','store_id'], how='left')

df = df.sort_values(by=['store_id','calendar_dt',]).reset_index(drop=True)

In [ ]:
pd.notna(df).sum()

Пропусков в колонках не так много пропусков, поэтому заменим их на локальную медиану (медиану по магазину, а не по всей выборке)

После предыдущей обработки остались пропущенные значения, это значения, где у магазинов вообще не было информации по этим критериям, поэтому их заменим их на медиану по всем магазинам

В fact_percent_lateness_lag_1 пропусков много, поэтому пока оставим без изменений

In [ ]:
for i in ['fact_couriers_with_shifts_lag_1', 'fact_load_factor_lag_1', 'predicted_num_orders', 'predicted_load_factor', 'predicted_staff_value']:
    df[i] = df[i].fillna(df.groupby('store_id')[i].transform('median'))
    df[i] = df[i].fillna(df[i].median())

df['marketing_costs_lag_1'] = df['marketing_costs_lag_1'].fillna(0)
df['predicted_load_factor'] = df['predicted_load_factor'].fillna(df.groupby('store_id')['predicted_load_factor'].transform('median'))

Создадим новые признаки, которые помогут Catboost лучше обучиться на данных:
- Извлечем номера месяцов и недель, 
- Создадим новый признак: если в проценте опозданий за пред. неделю стоит NaN, то значение будет 1, иначе - 0. 
- А в исходной колонке заменим значения NaN на -1. 
- Создадим разницы между предсказанными значениями и фактическими (и их процентные соотношения). В теории, это должно помочь найти зависимости
- Создадим лаги на 2 недели, чтобы модель увидела тенденции
- Добавим скользящее среднее
- Заменим пропуски на невозможное число, чтобы модель точно заметила их

In [ ]:
df['month'] = df['calendar_dt'].dt.month
df['week'] = df['calendar_dt'].dt.isocalendar().week.astype(int)

df['lateness_missing'] = df['fact_percent_lateness_lag_1'].isna().astype(int)
df['fact_percent_lateness_lag_1'] = df['fact_percent_lateness_lag_1'].fillna(-1)

df['delta_staff_value'] = df['predicted_staff_value'] - df['fact_staff_value_lag_1']
df['delta_num_oreders'] = df['predicted_num_orders'] - df['fact_num_orders_lag_1']
df['delta_load_factor'] = df['predicted_load_factor'] - df['fact_load_factor_lag_1']

df['delta_staff_value_percent'] = (df['predicted_staff_value'] - df['fact_staff_value_lag_1']) / df['fact_staff_value_lag_1']
df['delta_num_orders_percent'] = (df['predicted_num_orders'] - df['fact_num_orders_lag_1']) / df['fact_num_orders_lag_1']
df['delta_load_factor_percent'] = (df['predicted_load_factor'] - df['fact_load_factor_lag_1']) / df['fact_load_factor_lag_1']
df.replace([np.inf, -np.inf], np.nan, inplace=True)

# df['target_lag1'] = df.groupby('store_id')['target'].shift(1)
# df['target_lag2'] = df.groupby('store_id')['target'].shift(2)

df['fact_num_orders_lag_2'] = df.groupby('store_id')['fact_num_orders_lag_1'].shift(1)
df['fact_staff_value_lag_2'] = df.groupby('store_id')['fact_staff_value_lag_1'].shift(1)
df['fact_load_factor_lag_2'] = df.groupby('store_id')['fact_load_factor_lag_1'].shift(1)
df['fact_percent_lateness_lag_2'] = df.groupby('store_id')['fact_percent_lateness_lag_1'].shift(1)

# df['target_rolling'] = df.groupby('store_id')['target_lag1'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['fact_num_orders_rolling'] = df.groupby('store_id')['fact_num_orders_lag_1'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['fact_staff_value_rolling'] = df.groupby('store_id')['fact_staff_value_lag_1'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['fact_load_factor_rolling'] = df.groupby('store_id')['fact_load_factor_lag_1'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())
df['fact_percent_lateness_rolling'] = df.groupby('store_id')['fact_percent_lateness_lag_1'].transform(lambda x: x.rolling(window=3, min_periods=1).mean())

cols = ['fact_num_orders_lag_2','fact_staff_value_lag_2','fact_load_factor_lag_2','fact_percent_lateness_lag_2',
        'fact_num_orders_rolling','fact_staff_value_rolling','fact_load_factor_rolling','fact_percent_lateness_rolling']
for i in cols:
    df[i] = df[i].fillna(-1)

Создадим тестовую и тренировочную выборки, категариальные и числовые признаки.

In [ ]:
df['city_nm'] = df['city_nm'].astype('category')

train_data = df[df['calendar_dt'] < '2025-11-24'].copy()
test = df[df['calendar_dt'] == '2025-11-24'].copy()

num_features = [i for i in df.columns if i not in ['calendar_dt', 'store_id', 'target']]
cat_features = ['city_nm']

x = train_data[num_features]
y = train_data['target']
xtest = test[num_features]

print(f'Train shape: {x.shape}')
print(f'Test shape: {xtest.shape}')

Я решил попробовать использовать kfold и качество выросло. После подбора различных параметров, лучшими оказались эти:

In [ ]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

kf = KFold(n_splits=5, shuffle=True, random_state=42)

predictions = np.zeros(len(xtest))
wape_list = []

for fold, (train_id, val_id) in enumerate(kf.split(x)):
    xtrain, ytrain = x.iloc[train_id], y.iloc[train_id]
    xval, yval = x.iloc[val_id], y.iloc[val_id]
    
    model = lgb.LGBMRegressor(
        n_estimators=1500,
        learning_rate=0.03,
        num_leaves=22,
        max_depth=5,
        objective='MAE',
        metric='MAE',
        random_state=42,
        n_jobs=-1,
        verbose=-1
    )
    
    model.fit(
        X=xtrain,
        y=ytrain,
        eval_set=[(xval, yval)],
        eval_metric='MAE',
        categorical_feature=cat_features,
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    val_pred = model.predict(xval)
    val_pred = np.clip(val_pred, 0, None) 
    
    wape_fold = wape(yval.values, val_pred)
    print(f"Fold {fold+1} WAPE: {wape_fold:.4f}")
    wape_list.append(wape_fold)
    
    predictions += model.predict(xtest) / kf.n_splits

print(f"Средняя WAPE: {np.mean(wape_list):.4f}")

Загруэаем предсказание в файл

In [ ]:
test_pred = np.clip(predictions, 0, None)

answer = pd.DataFrame({
    'store_id':test['store_id'],
    'target':np.round(test_pred).astype(int) 
})

answer.to_csv('answer_gbm.csv', index=False)
print('Файл готов')

In [ ]:
answer